# DMRC RAG — 02. Gemma 2 9B Inference & API Serving

Requires a **GPU runtime** (Colab: Runtime -> Change runtime type -> GPU, ideally an A100/L4
with >=24GB VRAM for bf16; a T4 works with `GEMMA_USE_4BIT=1`).

Run `01_Setup_and_Retrieval_Validation.ipynb` first (or at least the clone + install cells
below) so `chroma_db/` and the retrieval stack are in place.

**Architecture note — why this uses `google/gemma-2-9b-it`, not Gemma 3 12B:**
the task here is *extractive* QA over retrieved contract clauses -- short, grounded context,
low need for creative generation -- which is exactly where a 7-9B instruct model performs
close to a 12B one. `src/gemma_inference.py` in the repo already reflects this: it loads
Gemma 2 9B with the plain `AutoModelForCausalLM` + `AutoTokenizer` pair (Gemma 3's 4B/12B/27B
checkpoints are vision-language models under a different loading path,
`Gemma3ForConditionalGeneration` + `AutoProcessor`, so this is a real architecture swap, not
just a smaller checkpoint of the same thing). This notebook uses that file as-is -- it does
**not** patch/overwrite it with the old Gemma 3 version.

### 1. Clone / update the repo

In [1]:
%cd /content
!test -d dmrc && (echo "dmrc/ already present -- pulling latest" && cd dmrc && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc.git
%cd /content/dmrc


/content
dmrc/ already present -- pulling latest
Already up to date.
/content/dmrc


### 2. Install dependencies

In [2]:
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" requirements.txt > /tmp/requirements_core.txt
!pip install -q -r /tmp/requirements_core.txt

# Best-effort only -- needed solely for GEMMA_USE_4BIT=1 later in this notebook.
!pip install -q bitsandbytes>=0.43.0 nvidia-nvjitlink-cu13 \
    || echo "4-bit extras failed to install -- fine if you're staying on the default bf16 path (GEMMA_USE_4BIT=0)."



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 21.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 137.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.9/434.9 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

### 3. Confirm GPU + CUDA

In [2]:
import torch
assert torch.cuda.is_available(), "No GPU detected -- switch this runtime to GPU before continuing."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


### 4. Hugging Face login

`google/gemma-2-9b-it` is a gated checkpoint -- you need to have accepted its license on
Hugging Face with the account you log in with here.

In [3]:
from huggingface_hub import login
login()


### 5. Load Gemma 2 9B and smoke-test `generate_answer`

First call is the slow one (weights download + load onto GPU); everything after reuses the
cached model via `gemma_inference.py`'s module-level cache.

In [4]:
from src.gemma_inference import get_gemma_model, generate_answer

model, tokenizer, device = get_gemma_model()
print(f"Model loaded. device={device}")

answer = generate_answer("What is Artificial Intelligence?")
print("\n" + answer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Model loaded. device=cuda
[gemma2] 14 prompt tok -> 320 new tok in 28.1s (11.4 tok/s)

Artificial intelligence (AI) is a broad field of computer science focused on creating machines capable of performing tasks that typically require human intelligence. 

Here's a breakdown:

**What AI aims to do:**

* **Learn:** AI systems can learn from data, identifying patterns and relationships.
* **Reason:** AI can use logic and rules to draw conclusions and make decisions.
* **Problem-solve:** AI can find solutions to complex problems, often by exploring multiple possibilities.
* **Perceive:** AI can interpret sensory information like images, sound, and text.
* **Understand and generate language:** AI can understand human language and generate its own text.

**Types of AI:**

* **Narrow or Weak AI:** Designed to perform a specific task, like playing chess or recommending products. Most AI today falls into this category.
* **General or Strong AI:** Hypothetical AI with human-level intelligence and

### 6. Release this notebook's copy of the model before starting the server

The `uvicorn` server started below runs in a **separate OS process** and loads its own copy
of Gemma 2 9B on the same GPU. Two resident copies (plus the embedding + reranker models) is
unnecessary VRAM pressure if you don't also need the model loaded here -- so free it first.

In [5]:
import gc, torch

for _name in ("model", "tokenizer"):
    if _name in globals():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
print("Notebook-side model reference cleared; GPU memory released.")


Notebook-side model reference cleared; GPU memory released.


### 7. Retrieval caps

Clause-number queries hit an exact-match fast path (one chunk, fast). Free-text queries run
the full retrieval pipeline and, uncapped, can hand the LLM a very large prompt -- prefill
cost grows quadratically with prompt length, so that's what actually causes timeouts, not the
model itself. This writes a small side-effect module that wraps `hybrid_search()` and
`rerank()` in place so `/ask` gets bounded output without touching `app.py`.

In [6]:
%%writefile src/retrieval_caps.py
"""
Runtime caps on retrieval breadth.

Imported for its side effects: wraps hybrid_search() and rerank() in place so
every caller -- including app.py's /ask endpoint -- gets bounded output without
any change to app.py itself.
"""
import os
import src.hybrid_retriever as hybrid_retriever
import src.reranker as reranker

MAX_CANDIDATES = int(os.environ.get("RAG_MAX_CANDIDATES", "20"))  # into reranker
MAX_CONTEXT    = int(os.environ.get("RAG_MAX_CONTEXT", "4"))      # into the LLM

_orig_hybrid = hybrid_retriever.hybrid_search
_orig_rerank = reranker.rerank


def _capped_hybrid(*args, **kwargs):
    hits = _orig_hybrid(*args, **kwargs)
    if hits and len(hits) > MAX_CANDIDATES:
        print(f"[cap] hybrid_search {len(hits)} -> {MAX_CANDIDATES}", flush=True)
        hits = hits[:MAX_CANDIDATES]
    return hits


def _capped_rerank(*args, **kwargs):
    out = _orig_rerank(*args, **kwargs)
    if out and len(out) > MAX_CONTEXT:
        print(f"[cap] rerank {len(out)} -> {MAX_CONTEXT}", flush=True)
        out = out[:MAX_CONTEXT]
    return out


hybrid_retriever.hybrid_search = _capped_hybrid
reranker.rerank = _capped_rerank


Writing src/retrieval_caps.py


In [7]:
# Apply the caps in THIS kernel and confirm the prompt shrinks.
import src.retrieval_caps  # noqa: F401  (imported for side effects)
import src.hybrid_retriever as hr
import src.reranker as rr
import src.prompt_engineering as pe
import time

QUERY = "What are the contractor obligations?"
t0 = time.time()
hits = hr.hybrid_search(QUERY)
reranked = rr.rerank(QUERY, hits)
prompt = pe.build_prompt(QUERY, reranked)
print(f"After caps: {len(reranked)} chunks, {len(prompt)} chars "
      f"(~{len(prompt)//4} tokens) in {time.time()-t0:.2f}s")
print("Target is roughly 1500-3000 tokens.")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loading embedding model: BAAI/bge-m3 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

[cap] rerank 5 -> 4
After caps: 4 chunks, 3904 chars (~976 tokens) in 34.67s
Target is roughly 1500-3000 tokens.


### 8. Make the caps load automatically inside the server subprocess

`uvicorn src.app:app` runs as a **separate process** from this kernel, so the caps applied in
cell 7 above don't carry over on their own. `sitecustomize.py` is auto-imported by Python at
startup whenever the current directory is on `sys.path` -- writing it here (plus setting
`PYTHONPATH` when we launch uvicorn, in the next cell) gets the caps into the server without
editing `app.py`.

In [8]:
%%writefile sitecustomize.py
"""Auto-imported by Python at startup when CWD is on sys.path.
Ensures the uvicorn subprocess picks up the retrieval caps without editing app.py.
"""
try:
    import src.retrieval_caps  # noqa: F401
except Exception as e:
    print(f"[sitecustomize] caps not loaded: {e}")


Writing sitecustomize.py


### 9. Start the FastAPI server

`GEMMA_USE_4BIT=0` -> full bf16, which is 3-5x faster than 4-bit (NF4) on an A100 with enough
VRAM headroom now that step 6 freed the notebook's own copy. Set it to `1` instead if you're
on a smaller GPU (e.g. a T4) and need the ~7-8GB 4-bit footprint.

In [9]:
import subprocess, time, os, requests

server_env = {
    **os.environ,
    "GEMMA_USE_4BIT": "0",
    "GEMMA_MAX_NEW_TOKENS": "320",
    "RAG_MAX_CANDIDATES": "20",     # read by src/retrieval_caps.py
    "RAG_MAX_CONTEXT": "4",
    "PYTHONPATH": os.getcwd(),      # so sitecustomize.py is found
}

server = subprocess.Popen(
    ["uvicorn", "src.app:app", "--host", "127.0.0.1", "--port", "8000"],
    env=server_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("Starting server (bf16, caps active)...")
start = time.time()
while True:
    if server.poll() is not None:
        print("Server exited early:")
        print(server.stdout.read())
        break
    try:
        r = requests.get("http://127.0.0.1:8000/status", timeout=3)
        if r.status_code == 200:
            print(f"HTTP up after {time.time()-start:.0f}s -> {r.json()}")
            break
    except requests.exceptions.RequestException:
        pass
    print(f"  ...{time.time()-start:.0f}s")
    time.sleep(5)


Starting server (bf16, caps active)...
  ...0s
  ...5s
  ...10s
  ...15s
  ...20s
  ...25s
  ...30s
  ...35s
HTTP up after 40s -> {'status': 'running', 'embedding_model': 'BAAI/bge-m3', 'reranker_model': 'BAAI/bge-reranker-v2-m3', 'dense_model_loaded': True, 'reranker_model_loaded': True, 'chromadb_connected': True}


### 10. Warm up

First `/ask` call includes the full model load inside the server process -- run this once before timing anything.

In [10]:
print("Warming up (first call includes the full model load)...")
t0 = time.time()
try:
    w = requests.post("http://127.0.0.1:8000/ask", json={"query": "warmup"}, timeout=1800)
    print(f"Warm-up done in {time.time()-t0:.0f}s -- status {w.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Warm-up failed after {time.time()-t0:.0f}s: {e}")
print("\nModel resident. Timings below are pure inference.")


Warming up (first call includes the full model load)...
Warm-up done in 3s -- status 200

Model resident. Timings below are pure inference.


### 11. Ask real questions

In [11]:
QUERIES = [
    "What are the contractor obligations?",
    "Who is responsible for maintenance during the defects liability period?",
    "What training must the contractor provide to employer staff?",
    "What are the requirements for the environmental control system software?",
    "Which stations are covered under this contract?",
]

results = []
for q in QUERIES:
    t0 = time.time()
    try:
        r = requests.post("http://127.0.0.1:8000/ask", json={"query": q}, timeout=300)
        dt = time.time() - t0
        if r.status_code == 200:
            data = r.json()
            src = data.get("sources", [])
            how = src[0].get("retrieval_source") if src else "-"
            print(f"\n{'='*74}\nQ: {q}\n   [{dt:.1f}s | {len(src)} sources | path: {how}]")
            print(f"\n{data.get('answer','')}")
            results.append((q, dt, True))
        else:
            print(f"\nQ: {q}\n   HTTP {r.status_code}: {r.text[:200]}")
            results.append((q, dt, False))
    except requests.exceptions.RequestException as e:
        dt = time.time() - t0
        print(f"\nQ: {q}\n   FAILED after {dt:.0f}s: {type(e).__name__}")
        results.append((q, dt, False))

print(f"\n\n{'='*74}\nSUMMARY")
for q, dt, ok in results:
    print(f"  {'OK ' if ok else 'FAIL'} {dt:6.1f}s  {q[:56]}")
ok_times = [d for _, d, o in results if o]
if ok_times:
    print(f"\n  {len(ok_times)}/{len(results)} succeeded | avg {sum(ok_times)/len(ok_times):.1f}s")



Q: What are the contractor obligations?
   [26.9s | 4 sources | path: dense+sparse]

The Contractor has several obligations as detailed in the provided Scope of Work:

* **Interfacing with Authorities:** Obtain necessary clearances or certificates from local authorities to ensure the ECS system's full functionality and operation as per the Contract. (Clause 4.2, Page 4)
* **Submission Management:** Maintain an organized file of all submissions, including an index and locating system to track their status. This system must be available for the Employer's Representative's review and used to assign sequential numbers to deliverables and track resubmissions. (Clause 6.10.3, Page 8)
* **Spare Part Testing:** Ensure all spares are correctly calibrated, tested, and labelled before delivery. Provide test certificates for each equipment to the Employer's Representative. (Clause 6.8.8, Page 5)
* **Method Statement and Drawings:** Submit a Method Statement to the Engineer for review at least 15 

### 12. Shut down

Run this last, after all `/ask` queries are done.

In [12]:
server.terminate()
try:
    server.wait(timeout=30)
    print("Server stopped.")
except subprocess.TimeoutExpired:
    server.kill()
    server.wait(timeout=10)
    print("Server didn't exit gracefully in 30s; force-killed it.")


Server stopped.
